# Reading Grape City Data

The data in https://github.com/Clear-Bible/grapecity-alignments comes from Grape City, a predecessor to Clear Bible. 

This uses a JSON format that combines sources, targets, and alignments into one (very large) JSON file and predates our current Scripture Burrito standard. But until all the data has be 🌯ified, this format remains relevant. 

The code in src.format purports to process this data: but i've forgotten how much of it works, it bundles in a lot of configuration logic that's no longer standard, and it's a legacy format. This is my attempt to simplify the process of extracting some of the data.

This example builds the MCL data for Burmese (`mya`). 

## Get Target TSV

The code in `src.gc2sb` assumes data is in `internal-alignments`. While that's no longer our standard (we use language-specific repositories instead), data is staged there temporarily while creating Burrito versions, and afterwards you can move it to the appropriate repository. 

In [1]:
# read the data from the legacy repository
from src.format import GRAPECITYDIR, gcreader
mcljson = GRAPECITYDIR / "mya/mclnt.nt.alignment.json"
gcr = gcreader.GCReader(mcljson)

In [2]:
# gcr is a dict from BCV strings to VerseData instances
list(gcr.items())[23]

('40002001', <VerseData(40002001)>)

In [3]:
# targets, converted to current standard
gcr.targetitems[:10]

[<Target: 40001001001>,
 <Target: 40001001002>,
 <Target: 40001001003>,
 <Target: 40001001004>,
 <Target: 40001001005>,
 <Target: 40001001006>,
 <Target: 40001001007>,
 <Target: 40001001008>,
 <Target: 40001001009>,
 <Target: 40001001010>]

In [5]:
gcr.legacyoutputfields


('id', 'altId', 'text', 'transType', 'isPunc', 'isPrimary')

In [4]:
# write them out to internal-alignments
# this uses y/n instead of True/False and omits some default values,
# but should otherwise be comparable to the pre-Burrito format
from src.burrito import DATAPATH, TargetReader
TargetReader.write_tsv(tokenlist=gcr.targetitems,
    outpath=(DATAPATH / f"targets/mya/MCL/nt_MCL.tsv"),
    fields=gcr.legacyoutputfields)

`nt_MCL.tsv` also appears to have around 6100 duplicates :-(
There are about 7150 verses covered: that's short about 800 relative to SBLGNT :-( (the same is true for JVB)

## Get Alignments

If the alignments are no longer present in pre-Burrito format.

TBD

## Upgrade the Data to Burrito Format



In [ ]:
from src.gc2sb import Manager, ManagerConfig
# legacy data should have reasonable entries in internal-Alignments/src/format/managerconfig.toml
# if not, update the file
mcfg = ManagerConfig()
# this assumes files are in expected places in internal-alignments
jvbcfg = mcfg.get_config("JVB", "nt_source")
mgr = Manager(jvbcfg)
agroup = mgr.read_gc()

In [ ]:
# this writes them to expected places in internal-alignments
mgr.write_burrito(agroup=agroup)

## Wrapping Up

`internal-alignments` should now include
- pre-Burrito target TSV
- current Burrito alignments

While it might be nice to upgrade the target TSV to the latest standard, it may not be necessary. Eventually a kathairo version will likely replace it: it's only useful for upgrading e.g., NA28 data to SBLGNT (maybe).

Next steps:
- copy the files to the appropriate `alignments-{lang}` repository
- let Rick or whomever know (if an upgrade to WLCM/SBLGNT is needed, which is likely)
- drop them from `internal-alignments` to preserve a Single Source of Truth(tm)
- read the data using `src.burrito.Manager` to ensure everything is okay

## Important Caveats!!

Running through this process for the `mya` JVB NT isn't yet working. 
- There are duplicate token IDs
    - The 12 tokens for 40003002 are repeated, the second time with -2 prefixes on `altId`
    - For 40003012, the tokens are repeated but out of order
- there's some indication that the alignment links cross verse boundaries in unexpected ways that violate the expectations of the code used above.

As of 2024-10-02 i'm deferring work on this data and turning to greener Burmese pastures.

See https://biblica.slack.com/archives/C07NV02F4BZ/p1727883672258379 for details.